# LLM MIA example: EZ-MIA on GPT-2 / WikiText-103

This notebook is the small, fast demo path: tokenise WikiText-103, fine-tune GPT-2, and audit it with
EZ-MIA -- fast enough to run interactively in notebook cells (a few minutes on a GPU/HPU). This folder
is EZ-MIA-only by design -- WBC lives in `../wbc/` (Pythia-2.8B / Khan Academy), which stays
script-based since a real Pythia-2.8B fine-tune takes far longer than a live cell should:
```bash
cd ../wbc
python ../prepare_target.py --config train_config_wbc_pythia.yaml   # or import_external_target.py if
                                                                       # you already have a checkpoint
python ../run_audit.py --audit audit_wbc_pythia_paper.yaml
```
See `README.md` for details on both paths.

In [ ]:
import os
import sys

import yaml

# llm_mia_main.ipynb lives in examples/mia/llm_mia/ez-mia/; the shared .py modules
# (hf_wrapper.py, llm_model_handler.py, prepare_target.py, etc.) live one level up.
llm_mia_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, llm_mia_dir)

project_root = os.path.abspath(os.path.join(os.getcwd(), '../../../..'))
sys.path.insert(0, project_root)

Load `train_config.yaml`, tokenise WikiText-103 (or reuse an already-tokenised population from a
previous run), and build the population dataset. This mirrors `prepare_target.py`'s own population step,
reusing the same helper functions rather than duplicating them.

In [ ]:
import random
from pathlib import Path

import joblib
import numpy as np
import torch
from transformers import AutoTokenizer

from prepare_target import _load_texts, _tokenise
from llm_data_handler import LLMDataHandler

with open('train_config_ez-mia.yaml') as f:
    train_config = yaml.safe_load(f)
run_cfg, train_cfg, data_cfg = train_config['run'], train_config['train'], train_config['data']

random.seed(run_cfg['random_seed'])
tokenizer = AutoTokenizer.from_pretrained(train_cfg['model_name'])
pad_token_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id

data_path = Path(data_cfg['data_path'])
if data_path.exists():
    population = joblib.load(data_path)
    print(f"loaded population from {data_path} ({len(population)} sequences)")
else:
    texts = _load_texts(data_cfg)
    random.shuffle(texts)
    seqs = _tokenise(texts, tokenizer, data_cfg)
    if len(seqs) < data_cfg['n_sequences']:
        raise ValueError(f"only {len(seqs)} sequences available, need {data_cfg['n_sequences']}")
    ids = torch.as_tensor(np.stack(seqs)) if data_cfg['chunking'] == 'fixed' else LLMDataHandler.as_object_array(seqs)
    population = LLMDataHandler.UserDataset(ids, ids, pad_token_id=pad_token_id)
    data_path.parent.mkdir(parents=True, exist_ok=True)
    joblib.dump(population, data_path)
    print(f"saved population to {data_path} ({len(population)} sequences)")

Split the population into a fine-tuning (member) set and a held-out (non-member) set. The same seed
`prepare_target.py` uses, so re-running this notebook reproduces the identical split.

In [ ]:
n = len(population)
n_members, n_nonmembers = data_cfg['n_members'], data_cfg['n_nonmembers']
assert n_members + n_nonmembers <= n, "population too small for the requested member/non-member split"
perm = np.random.RandomState(run_cfg['random_seed']).permutation(n)
train_indices = perm[:n_members].tolist()
test_indices = perm[n_members:n_members + n_nonmembers].tolist()
print(f"{len(train_indices)} members, {len(test_indices)} non-members")

Fine-tune GPT-2 as the target model. `LLMModelHandler.train` supports gradient accumulation, LR
warmup, and per-epoch held-out eval if `train_config.yaml` asks for them (see the README's
"Reproducing a specific paper config" section) -- this demo config uses none of those, just a plain
3-epoch fine-tune.

In [ ]:
from torch import nn, optim
from torch.utils.data import DataLoader

from hf_wrapper import HFCausalLMWrapper
from llm_model_handler import LLMModelHandler
from leakpro.signals.token_evidence import CausalLMCollate

collate = CausalLMCollate(pad_token_id=pad_token_id)
train_loader = DataLoader(
    LLMDataHandler.UserDataset(population.data[train_indices], population.targets[train_indices], pad_token_id=pad_token_id),
    batch_size=train_cfg['batch_size'], shuffle=True, collate_fn=collate)
test_loader = DataLoader(
    LLMDataHandler.UserDataset(population.data[test_indices], population.targets[test_indices], pad_token_id=pad_token_id),
    batch_size=train_cfg['batch_size'], shuffle=False, collate_fn=collate)

model = HFCausalLMWrapper(train_cfg['model_name'], dtype=train_cfg.get('dtype', 'float32'))
criterion = nn.CrossEntropyLoss(ignore_index=-100)
optimizer = optim.AdamW(model.parameters(), lr=train_cfg['learning_rate'], weight_decay=train_cfg.get('weight_decay', 0.0))

LLMModelHandler.lora = train_cfg['lora'] if train_cfg['finetune_method'] == 'lora' else None
handler = LLMModelHandler()
train_result = handler.train(
    train_loader, model, criterion, optimizer, epochs=train_cfg['epochs'],
    gradient_accumulation_steps=train_cfg.get('gradient_accumulation_steps', 1),
    warmup_steps=train_cfg.get('warmup_steps', 0),
    eval_dataloader=test_loader if train_cfg.get('eval_strategy') == 'epoch' else None,
    checkpoint_dir=None,  # notebook demo: keep only the final fine-tuned target, no per-epoch checkpoints
    save_total_limit=train_cfg.get('save_total_limit', 1),
)
test_result = handler.eval(test_loader, train_result.model, criterion)
print(f"held-out next-token acc {test_result.accuracy:.4f}  loss {test_result.loss:.4f}")

Plot the target model's training/held-out curves.

In [ ]:
import matplotlib.pyplot as plt

extra = train_result.metrics.extra
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.plot(extra['loss_history'], label='Train loss')
if extra['val_loss_history']:
    plt.plot(extra['val_loss_history'], label='Held-out loss')
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.legend()

plt.subplot(1, 2, 2)
plt.plot(extra['acc_history'], label='Train next-token acc')
if extra['val_acc_history']:
    plt.plot(extra['val_acc_history'], label='Held-out next-token acc')
plt.xlabel('Epoch'); plt.ylabel('Accuracy'); plt.legend()

plt.tight_layout()
plt.show()

Save the target model and the metadata LeakPro's MIA handler needs -- the same two files
`prepare_target.py` writes, so `audit.yaml` (pointing at `target_full`) needs no changes.

In [ ]:
import pickle

from leakpro import LeakPro

log_dir = Path(run_cfg['log_dir'].format(finetune_method=train_cfg['finetune_method']))
log_dir.mkdir(parents=True, exist_ok=True)

model = train_result.model
model.to('cpu')
with open(log_dir / 'target_model.pkl', 'wb') as f:
    torch.save({k: v.cpu() for k, v in model.state_dict().items()}, f)

metadata = LeakPro.make_mia_metadata(
    train_result=train_result, optimizer=optimizer, loss_fn=criterion, dataloader=train_loader,
    test_result=test_result, epochs=train_cfg['epochs'], train_indices=train_indices,
    test_indices=test_indices, dataset_name=data_cfg['hf_dataset'],
)
with open(log_dir / 'model_metadata.pkl', 'wb') as f:
    pickle.dump(metadata, f)
print(f"wrote target to {log_dir}")

## Privacy Auditing using LeakPro

1. Create a LeakPro instance with separate data and model handlers.
2. Run the attack defined in `audit.yaml` (EZ-MIA).
3. Inspect the result object.

In [ ]:
from leakpro import LeakPro
from llm_data_handler import LLMDataHandler
from llm_model_handler import LLMModelHandler

leakpro = LeakPro(LLMDataHandler, 'audit.yaml', model_handler=LLMModelHandler)
mia_results = leakpro.run_audit(create_pdf=True)
for res in mia_results:
    print(f"{res.result_name}: AUC {res.roc_auc:.4f}  " + "  ".join(f"{k} {v}" for k, v in res.fixed_fpr_table.items()))